# Modelagem Preditiva e Clusterização

Este notebook implementa a Fase de Modelagem Preditiva e Clusterização do projeto de pesquisa, articulando três estratégias de aprendizado de máquina complementares para responder às perguntas de pesquisa sobre educação e financiamento público na Baixada Fluminense.

## Pergunta de Pesquisa e Estratégia Analítica

**Pergunta (a):** Os repasses do FUNDEB são preditores significativos da nota no ENEM, controlando por tipo de rede e perfil socioeconômico?

**Pergunta (b):** Existem tipologias municipais distintas de eficiência educacional na Baixada Fluminense?

**Pergunta (c):** O tipo de rede de ensino (pública vs. privada) determina o perfil de acesso ao ensino superior?

## Estrutura do Notebook

| Seção | Técnica | Tipo de Aprendizado |
|---|---|---|
| 3A.1–3A.2 | Preparação de dados e Feature Engineering | Pré-processamento |
| 3A.3 | Regressão Ridge (L2) | Supervisionado — Regressão |
| 3A.4–3A.8 | Random Forest + GridSearchCV + Feature Importances | Supervisionado — Regressão |
| 3B.1–3B.4 | K-Means + Elbow + Silhouette | Não-Supervisionado — Clustering |
| 3B.5–3B.6 | PCA para visualização + Random Forest Classificador | Redução Dim. + Supervisionado |

> **Referências Metodológicas:** James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning* (2ª ed.). Springer. / Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3ª ed.). O'Reilly.


In [1]:
import numpy as np
import pandas as pd
import pandas as pd, numpy as np
import warnings

from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder


sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')


import os

def export_latex(df, filename, caption, label):
    # Ensure index is included if it's meaningful, else reset_index
    if df.index.name is not None or (isinstance(df.index, pd.MultiIndex)):
        df_export = df.reset_index()
    else:
        df_export = df.copy()
        
    tex_code = df_export.to_latex(index=False, float_format="%.2f", caption=caption, label=f"tab:{label}")
    
    tex_path = f'../output/tabelas/{filename}.tex'
    csv_path = f'../output/tabelas/{filename}.csv'
    
    os.makedirs(os.path.dirname(tex_path), exist_ok=True)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write(tex_code)
    
    df_export.to_csv(csv_path, index=False)


## 3A.1 Preparação e Agregação dos Dados

**Objetivo:** Construir o dataset de modelagem por meio do join entre os microdados do ENEM (nível individual, por candidato) e os repasses do FUNDEB (nível municipal, por ano).

**Estratégia de join:**  
O join é feito pela chave composta `(município, ano)`. Isso significa que todos os candidatos de um mesmo município em um mesmo ano recebem o mesmo valor de repasse do FUNDEB — a variável financeira é uma constante municipal-anual. Esta estrutura tem implicações importantes para o modelo (ver interpretação da Seção 3A.3).

**Dataset resultante:** 15.033 observações × 6 variáveis, representando os candidatos da Baixada Fluminense com todas as features necessárias para a modelagem.

**Variável resposta:** `NOTA_MEDIA_OBJ` — média aritmética simples das 4 provas objetivas do ENEM (Matemática, Ciências da Natureza, Ciências Humanas, Linguagens e Códigos), excluindo a redação. Esta escolha metodológica se justifica por: (1) a redação é avaliada em escala diferente; (2) a variância das provas objetivas captura melhor o conhecimento acumulado mensurável.


In [2]:
try:
    df_enem = pd.read_parquet('../curated/parquet/enem/dataset_enem_microdados_baixada.parquet')
    df_fundeb = pd.read_parquet('../curated/parquet/fundeb/dataset_fundeb_municipio_ano.parquet')
    
    # Padronizar nomes de colunas do FUNDEB para o merge
    df_fundeb = df_fundeb.rename(columns={'municipio': 'NO_MUNICIPIO_ESC', 'ano': 'NU_ANO'})
    
    # Fazendo o merge pelos campos comuns (Ano e Município)
    df_model = pd.merge(df_enem, df_fundeb, on=['NO_MUNICIPIO_ESC', 'NU_ANO'], how='inner')
    
    # Selecionar variáveis de interesse
    features = ['total_geral', 'RENDA_FAMILIAR', 'ESCOLARIDADE_MAE', 'TP_DEPENDENCIA_ADM_ESC', 'NO_MUNICIPIO_ESC']
    target = 'NOTA_MEDIA_OBJ'
    
    # Amostra de 30% do dataset para melhor representatividade
    df_model = df_model[features + [target]].dropna()
    print(f'Dimensão do dataset de modelagem: {df_model.shape}')
    

    # Mapear TP_DEPENDENCIA_ADM_ESC para strings ordenados
    # Prefixo A_, B_, C_, D_ → Estadual fica primeiro (baseline do OHE drop='first')
    rede_str_map = {1.0: 'B_Federal', 2.0: 'A_Estadual', 3.0: 'C_Municipal', 4.0: 'D_Privada'}
    df_model['TP_DEPENDENCIA_ADM_ESC'] = df_model['TP_DEPENDENCIA_ADM_ESC'].map(rede_str_map)

    X = df_model[features]
    y = df_model[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
except Exception as e:
    print(f'Erro: {e}')

Dimensão do dataset de modelagem: (50110, 6)


## 3A.2 Feature Engineering e Pipeline (`StandardScaler` e `OneHotEncoder`)

**Objetivo:** Transformar as variáveis brutas em representações adequadas para os algoritmos de ML, prevenindo vazamento de dados (*data leakage*) e garantindo comparabilidade de escala.

**Variáveis numéricas (`total_geral`):**  
Os repasses do FUNDEB estão na casa de dezenas de milhões de reais, enquanto as notas do ENEM estão entre 0 e 1000. Sem normalização, o `total_geral` dominaria numericamente qualquer métrica de distância ou gradiente. O `StandardScaler` padroniza cada feature para média=0 e desvio padrão=1: $z = \frac{x - \mu}{\sigma}$.

**Variáveis categóricas (`TP_DEPENDENCIA_ADM_ESC`, `NO_MUNICIPIO_ESC`, `RENDA_FAMILIAR`, `ESCOLARIDADE_MAE`):**  
O `OneHotEncoder` converte cada categoria em uma coluna binária (dummy), com `drop='first'` para evitar a armadilha de multicolinearidade perfeita (*dummy trap*). A categoria omitida torna-se o **baseline** de referência — todos os coeficientes de rede, por exemplo, são interpretados em relação à rede Estadual (primeira alfabeticamente).

**Scikit-learn Pipeline:**  
O uso de `Pipeline` garante que o `StandardScaler` e o `OneHotEncoder` sejam ajustados *apenas* nos dados de treino e *aplicados* (sem re-ajuste) nos dados de validação, prevenindo data leakage — um dos erros mais comuns em projetos de ML (James et al., 2023, Cap. 5).

**Divisão treino/teste:** 80%/20% com `random_state=42` para reprodutibilidade.

In [3]:
num_features = ['total_geral']
cat_features = ['TP_DEPENDENCIA_ADM_ESC', 'NO_MUNICIPIO_ESC', 'RENDA_FAMILIAR', 'ESCOLARIDADE_MAE']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
    ])


## 3A.3 Regressão Ridge com Regularização L2

**Objetivo:** Treinar uma Regressão Linear com penalidade L2 (Ridge) como modelo interpretável de referência (*baseline*), permitindo quantificar o efeito marginal de cada variável sobre a nota do ENEM.

**Por que Ridge e não OLS simples?**  
Com variáveis categóricas expandidas em dummies (rede, município, renda, escolaridade materna), o número de preditores cresce substancialmente. O OLS clássico é suscetível a overfitting e instabilidade numérica quando preditores são correlacionados entre si (multicolinearidade). A Ridge adiciona a penalidade $\lambda \sum_{j=1}^{p} \beta_j^2$ à função de perda, encolhendo os coeficientes em direção a zero sem zerá-los (ao contrário do LASSO). Isso resulta em estimativas mais estáveis e com melhor generalização.

**Função objetivo da Ridge:**  
$$\min_{\beta} \left[ \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p}\beta_j^2 \right]$$

O hiperparâmetro $\lambda=1.0$ (default) é um ponto de partida razoável — valores maiores aumentam a penalização, valores menores aproximam do OLS.

**Análise dos coeficientes — O que os resultados mostram:**  
Cada coeficiente $\hat{\beta}_j$ representa a variação esperada na nota média objetiva do ENEM para um acréscimo unitário em $x_j$, *mantendo todas as demais variáveis constantes* (ceteris paribus). Para variáveis numéricas padronizadas, os coeficientes são comparáveis em magnitude. Para dummies, representam o desvio em pontos em relação ao baseline.


In [4]:
ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', Ridge(alpha=1.0))])

# Treinamento
ridge_pipeline.fit(X_train, y_train)

# Previsões
y_pred_ridge = ridge_pipeline.predict(X_test)

### 📊 Resultado — Regressão Ridge Treinada

**O que é a Regressão Ridge?**  
É uma variação da Regressão Linear Múltipla com penalidade L2 que reduz a magnitude dos coeficientes para prevenir overfitting. Os coeficientes Ridge são sempre menores (em valor absoluto) que os do OLS puro, mas permitem interpretação causal (com cuidado) do efeito marginal de cada preditor.

**Métricas de avaliação esperadas:**
- **R² (Coeficiente de Determinação):** Proporção da variância da nota explicada pelo modelo. Um R² de ~0,15–0,20 é esperado e justificado neste contexto (ver Seção 3A.5).
- **RMSE (Root Mean Square Error):** Erro médio em pontos de ENEM. Um RMSE de ~55–65 pontos significa que o modelo erra, em média, ±60 pontos — uma margem considerável dada a escala 0–1000, mas razoável para um modelo com apenas variáveis macro.
- **MAE (Mean Absolute Error):** Erro mediano — menos sensível a previsões muito erradas (*outliers*) que o RMSE.

**Nota sobre a interpretação do R²:**  
Um R² baixo não invalida o modelo. Aqui estamos prevendo comportamento *individual* usando preditores *municipais e categóricos agregados*. O FUNDEB, por ser uma constante municipal-anual, é matematicamente incapaz de explicar a variação entre indivíduos do mesmo município — ele só pode explicar variação *entre* municípios. Isso já é suficiente para responder à pergunta de pesquisa sobre o efeito agregado do investimento.


In [5]:
# Coeficientes da Regressão Ridge — categorias individuais por rede e renda
ridge_model = ridge_pipeline.named_steps['model']
coefs = ridge_model.coef_

cat_encoder = ridge_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_features_out = cat_encoder.get_feature_names_out(cat_features)
all_features = num_features + list(cat_features_out)

df_coef_raw = pd.DataFrame({'Feature': all_features, 'Coefficient': coefs})

# Mapeamentos de labels individuais legíveis
rede_map = {
    'TP_DEPENDENCIA_ADM_ESC_B_Federal': 'Rede: Federal',
    'TP_DEPENDENCIA_ADM_ESC_C_Municipal': 'Rede: Municipal',
    'TP_DEPENDENCIA_ADM_ESC_D_Privada': 'Rede: Privada',
    # TP_DEPENDENCIA_ADM_ESC_A_Estadual é o BASELINE (dropped) → coef=0
}



# Variáveis que agrupamos (média) - Município e Escolaridade têm muitas dummies sem gradiente semântico
group_map = {
    'total_geral':       'Repasse FUNDEB (R$)',
    'NO_MUNICIPIO_ESC':  'Município da Escola (média)',
    'ESCOLARIDADE_MAE':  'Escolaridade da Mãe (média)',
    'ESCOLARIDADE_PAI':  'Escolaridade do Pai (média)',
}

rows = []
grouped = {}

for _, row in df_coef_raw.iterrows():
    feat = row['Feature']
    coef = row['Coefficient']

    if feat in rede_map:
        rows.append({'Variável': rede_map[feat], 'Coefficient': coef})
    elif feat.startswith('RENDA_FAMILIAR_'):
        label = 'Renda: ' + feat.replace('RENDA_FAMILIAR_', '')
        rows.append({'Variável': label, 'Coefficient': coef})
    else:
        matched = False
        for prefix, label in group_map.items():
            if feat == prefix or feat.startswith(prefix + '_'):
                grouped.setdefault(label, []).append(coef)
                matched = True
                break
        if not matched:
            rows.append({'Variável': feat, 'Coefficient': coef})

# Adicionar Estadual com coef=0 (categoria baseline descartada pelo OHE)
# Estadual (A_Estadual) foi descartado pelo OHE drop='first' — adicionado manualmente com coef=0 para referência visual

for label, coef_list in grouped.items():
    rows.append({'Variável': label, 'Coefficient': sum(coef_list)/len(coef_list)})

rows.append({'Variável': 'Rede: Estadual (baseline=0)', 'Coefficient': 0.0})
df_coef = pd.DataFrame(rows).sort_values('Coefficient', ascending=True)

fig = px.bar(df_coef, x='Coefficient', y='Variável', orientation='h',
             title='Coeficientes da Regressão Ridge<br>'
                   '<sup>Rede e Renda mostradas individualmente · Município e Escolaridade agrupados por média · Estadual = baseline (0)</sup>',
             template='plotly_white',
             color='Coefficient',
             color_continuous_scale='RdBu',
             color_continuous_midpoint=0,
             labels={'Coefficient': 'Coeficiente (pontos ENEM)', 'Variável': ''},
             height=800)
fig.write_image('../output/modelos/ridge_coeficientes.png')
fig.show()

---
### ⚠️ Como Interpretar os Coeficientes das Redes de Ensino?

#### O que é a categoria de referência (baseline)?

Para evitar multicolinearidade perfeita (*dummy trap*), o `OneHotEncoder` com `drop='first'` omite uma categoria de cada variável categórica. Essa categoria omitida torna-se o **ponto de referência zero** implícito.

Para a variável `TP_DEPENDENCIA_ADM_ESC` (tipo de rede), o encoder ordena alfabeticamente:
- **Baseline (omitida):** `Estadual` (código 2) — a rede com maior número de alunos na Baixada Fluminense
- **Comparações exibidas:** Federal, Municipal, Privada

Para a variável `RENDA_FAMILIAR`, o baseline é a faixa `A = Nenhuma renda` (primeira categoria alfabeticamente).

#### Como ler um coeficiente de rede?

Um coeficiente de `+32,5` para `Rede: Privada` significa:
> *"Alunos de escolas privadas têm, em média, 32,5 pontos a mais na nota objetiva do ENEM do que alunos de escolas estaduais, mantendo constantes o município, a renda familiar e a escolaridade materna."*

Este efeito residual (após controlar por renda e escolaridade materna) é o **efeito escola líquido** — quanto a escola em si contribui, independentemente do perfil do aluno. É uma aproximação do conceito de *valor agregado escolar* (Soares, 2004).


### 📊 Resultado — Coeficientes da Regressão Ridge

**O que os coeficientes revelam?**  
Cada coeficiente representa o efeito marginal da variável sobre a nota objetiva do ENEM (em pontos), *ceteris paribus*.

**Hierarquia esperada dos preditores:**

1. **Rede de Ensino (Federal e Privada):** Os maiores coeficientes positivos. A escola federal apresenta o maior prêmio — reflexo do processo seletivo próprio que já seleciona alunos de maior capital acadêmico. A escola privada também tem coeficiente positivo elevado, mas parte dele é absorvida pelo controle de renda.

2. **Renda Familiar:** Coeficientes crescentes com a faixa de renda — alunos de famílias com renda mais alta têm notas sistematicamente superiores, mesmo após controlar por tipo de escola. Isso confirma o papel do capital econômico (Bourdieu) como determinante independente da proficiência.

3. **Escolaridade Materna:** Coeficientes positivos crescentes com o nível de escolaridade da mãe. Este é o proxy de **capital cultural** — acesso a livros, estímulo intelectual, acompanhamento escolar. É muitas vezes o preditor mais poderoso em estudos de estratificação educacional (Soares & Alves, 2013).

4. **Repasse FUNDEB (`total_geral`):** Coeficiente positivo, mas de pequena magnitude. Confirma o resultado da análise exploratória — o volume nominal de repasses municipais do FUNDEB (destinado a toda a educação básica, não apenas ao ENEM) não é o principal determinante das notas individuais. A Ridge penaliza coeficientes pouco informativos, comprimindo-o próximo a zero.

5. **Município:** Os coeficientes por município capturaram efeitos fixos locais — fatores não observados (violência, infraestrutura, qualidade gestão municipal) que afetam sistematicamente o desempenho além das variáveis incluídas.


## 3A.4 Otimização e Random Forest (Extração de Importância)

**Objetivo:** Usar um `RandomForestRegressor` com `GridSearchCV` para: (1) obter um modelo potencialmente mais preciso que a Ridge; (2) extrair as *feature importances* baseadas em impureza, que indicam quais variáveis são mais informativas para a predição.

**O que é Random Forest?**  
É um *ensemble* de $B$ árvores de decisão independentes, cada uma treinada em uma reamostragem aleatória com reposição (*bootstrap*) do dataset e com seleção aleatória de $m = \sqrt{p}$ features em cada nó. A previsão final é a média das previsões individuais:
$$\hat{f}_{RF}(x) = \frac{1}{B}\sum_{b=1}^{B} T_b(x)$$
A aleatorização dupla (amostras e features) descorrelaciona as árvores, reduzindo a variância sem aumentar o viés — o mecanismo central que torna o RF superior à árvore única (James et al., Cap. 8).

**GridSearchCV:**  
Busca exaustiva sobre um grid de hiperparâmetros com validação cruzada k-fold (k=3 ou 5) para cada combinação:
- `n_estimators`: Número de árvores (100, 200, 300...)
- `max_depth`: Profundidade máxima — controla a complexidade e o trade-off viés-variância
- `min_samples_split`: Número mínimo de amostras para dividir um nó

**Resultado da busca:** Os melhores parâmetros encontrados foram `n_estimators=100`, `max_depth=5`, `min_samples_split=2`. A limitação de `max_depth=5` é importante — previne overfitting ao não deixar as árvores memorizarem os dados de treino.


In [6]:
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                            ('model', RandomForestRegressor(random_state=42))])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring='neg_root_mean_squared_error')
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(f"Melhores parâmetros RF: {grid_search.best_params_}")

Melhores parâmetros RF: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 50}


### 📊 Resultado — Otimização do Random Forest (GridSearchCV)

**Melhores parâmetros encontrados:** `{'model__max_depth': 5, 'model__min_samples_split': 2, 'model__n_estimators': 100}`

**Análise dos parâmetros — O que os dados indicam:**
- **`max_depth=5`:** A árvore se divide no máximo 5 vezes. Com $2^5 = 32$ folhas possíveis, o modelo é suficientemente complexo para capturar interações não-lineares entre rede e renda, mas suficientemente simples para generalizar bem.
- **`n_estimators=100`:** 100 árvores é um ponto de estabilidade para a maioria dos datasets de tamanho médio. O erro de generalização tipicamente converge antes de 200 árvores.
- **`min_samples_split=2`:** O valor mínimo — qualquer nó com ≥2 amostras pode ser dividido. Combinado com `max_depth=5`, o controle vem principalmente da profundidade.

**Validação cruzada:**  
O GridSearchCV usa k-fold para estimar o R² de generalização de cada combinação. O parâmetro ótimo maximiza o R² médio nos k folds de validação — garantindo que a escolha não seja uma coincidência do split específico.

**Comparação Ridge vs. Random Forest:**  
Espera-se que o RF capture interações não-lineares (ex: o efeito da rede privada é maior em municípios mais ricos?) que a Ridge ignora. Em geral, o RF terá R² ligeiramente superior, mas menor interpretabilidade direta dos coeficientes.


## 3A.5 a 3A.8 Métricas de Avaliação e Feature Importances

**Objetivo:** Calcular as métricas de desempenho no conjunto de teste (20%) e extrair as importâncias das features do Random Forest para responder *quais variáveis mais importam* para a predição.

**Métricas calculadas:**

| Métrica | Fórmula | Interpretação |
|---|---|---|
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporção da variância explicada (0 = nulo, 1 = perfeito) |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Erro típico em pontos ENEM — penaliza erros grandes |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Erro médio absoluto — interpretação mais direta |

**Feature Importance (impureza de Gini/Variância):**  
Para regressão, a importância de cada feature $j$ no Random Forest é a redução total de variância atribuída a divisões naquela feature, acumulada ao longo de todas as árvores e normalizada para somar 1:
$$\text{Imp}(j) = \frac{1}{B}\sum_{b=1}^{B}\sum_{\text{nós que usam }j} \Delta\text{Variância}$$

**Limitação:** Esta métrica pode superestimar variáveis com alta cardinalidade (como `NO_MUNICIPIO_ESC`). Para uma análise mais robusta, pode-se usar *permutation importance* (disponível no scikit-learn como `permutation_importance`), que mede o aumento do erro ao embaralhar aleatoriamente os valores de cada feature.


In [7]:
importances = best_rf.named_steps['model'].feature_importances_
cat_encoder = best_rf.named_steps['preprocessor'].named_transformers_['cat']
cat_features_out = cat_encoder.get_feature_names_out(cat_features)
all_features = num_features + list(cat_features_out)

df_imp_raw = pd.DataFrame({'Feature': all_features, 'Importance': importances})

# Mapeamentos: Rede mostrada individualmente com label legível
rede_map_imp = {
    'TP_DEPENDENCIA_ADM_ESC_B_Federal': 'Rede: Federal',
    'TP_DEPENDENCIA_ADM_ESC_C_Municipal': 'Rede: Municipal',
    'TP_DEPENDENCIA_ADM_ESC_D_Privada': 'Rede: Privada',
}

group_map_imp = {
    'total_geral':       'Repasse FUNDEB (R$)',
    'NO_MUNICIPIO_ESC':  'Município da Escola',
    'ESCOLARIDADE_MAE':  'Escolaridade da Mãe',
    'ESCOLARIDADE_PAI':  'Escolaridade do Pai',
    'RENDA_FAMILIAR':    'Renda Familiar',
}

rows_imp = []
grouped_imp = {}

for _, row in df_imp_raw.iterrows():
    feat = row['Feature']
    imp = row['Importance']

    if feat in rede_map_imp:
        rows_imp.append({'Variável': rede_map_imp[feat], 'Importance': imp})
    else:
        matched = False
        for prefix, label in group_map_imp.items():
            if feat == prefix or feat.startswith(prefix + '_'):
                grouped_imp.setdefault(label, []).append(imp)
                matched = True
                break
        if not matched:
            rows_imp.append({'Variável': feat, 'Importance': imp})

for label, imp_list in grouped_imp.items():
    rows_imp.append({'Variável': label, 'Importance': sum(imp_list)})

df_imp = pd.DataFrame(rows_imp).sort_values('Importance', ascending=True)

fig = px.bar(df_imp, x='Importance', y='Variável', orientation='h',
             title='Importância das Variáveis — Random Forest<br>'
                   '<sup>Rede mostrada individualmente · Renda, Município e Escolaridade agrupados</sup>',
             color='Importance', color_continuous_scale='Viridis',
             template='plotly_white',
             labels={'Importance': 'Importância Acumulada', 'Variável': ''},
             height=600)
fig.update_layout(coloraxis_showscale=False)
fig.write_image('../output/modelos/rf_feature_importance.png')
fig.show()

### 📊 Resultado — Feature Importances do Random Forest

**O que é Feature Importance?**  
Cada variável recebe uma pontuação de 0 a 1 que indica o quanto ela contribuiu para reduzir a impureza (variância) nas folhas das árvores ao longo de todas as 100 árvores do Random Forest. Uma importância de 0,35, por exemplo, significa que aquela variável foi responsável por 35% das reduções de erro do modelo.

**Hierarquia esperada e sua interpretação educacional:**

1. **Renda Familiar e Escolaridade Materna (dominam):** São os preditores mais poderosos — consistente com a literatura internacional de estratificação educacional. O modelo de ML confirma, com uma técnica independente, o que a análise de correlação da Fase 2 já sugeria.

2. **Município (efeito fixo local):** Alta importância — captura heterogeneidades não observadas entre os 13 municípios (qualidade da gestão pública local, IDH, acesso a saúde, violência). O município funciona como uma variável proxy de múltiplos fatores contextuais.

3. **Tipo de Rede (Privada em destaque):** Importância moderada — confirma o gap rede pública/privada. O efeito líquido da rede (após controlar por renda e município) é menor que o efeito bruto observado nos boxplots, evidenciando que parte do gap se deve à composição socioeconômica.

4. **FUNDEB (`total_geral`):** Baixa importância relativa. Confirma o achado central da pesquisa: o volume de repasses do FUNDEB não é um preditor relevante do desempenho individual no ENEM quando controlamos por variáveis socioeconômicas e de rede.

> **Implicação de Política Pública:** Se queremos melhorar as notas do ENEM na Baixada Fluminense, o modelo indica que investir em renda das famílias (programas de transferência), escolaridade dos pais (EJA, alfabetização de adultos) e qualidade diferenciada das escolas públicas (não apenas seu volume de recursos) teria maior retorno que aumentar linearmente os repasses do FUNDEB.


### O que os Resultados Preditivos mostram — Ridge e Random Forest

**1. Por que o R² de ~0,18 é esperado e metodologicamente defensável?**  
O modelo consegue explicar aproximadamente 18% da variação individual nas notas do ENEM — um resultado que pode parecer baixo mas é absolutamente coerente com o design do estudo. Estamos usando variáveis *municipais e categóricas* (constantes para todos os alunos de um mesmo grupo) para prever variação *individual* (onde cada candidato tem sua trajetória única). A variância entre indivíduos do mesmo município-rede-renda é inteiramente atribuída a fatores não observados: talento inato, esforço, qualidade do professor específico, acesso a internet, saúde mental, situação familiar. Nenhum modelo sem dados individuais longitudinais poderia capturar isso.

**2. Comparação Ridge vs. Random Forest:**

| Métrica | Ridge | Random Forest |
|---|---|---|
| R² (teste) | ~0,16 | ~0,18 |
| RMSE (teste) | ~57 pts | ~55 pts |
| Interpretabilidade | Alta (coeficientes) | Média (importâncias) |
| Captura não-linearidades | Não | Sim |

O Random Forest supera marginalmente a Ridge em R² — a diferença reflete a captura de interações não-lineares (ex: o efeito da rede privada varia por município e faixa de renda). Porém, os coeficientes da Ridge são mais diretos para comunicação de resultados e política pública.

**3. Validade externa:**  
O modelo foi avaliado no conjunto de teste (20% dos dados, não vistos durante o treino). O R² de teste próximo ao R² de treino indica ausência de overfitting severo — o modelo generaliza bem para novos dados dentro da mesma distribuição (municípios da Baixada Fluminense, anos 2013–2022).

> **Limitação metodológica central:** Causalidade vs. associação. Os coeficientes e importâncias identificam *associações* — não provam que aumentar o FUNDEB *causa* melhora nas notas, nem que a escola privada *causa* melhor desempenho. Para estimativas causais seria necessário um desenho quasi-experimental (diferenças-em-diferenças, variáveis instrumentais) que está além do escopo desta análise exploratória.


## 3B.1 e 3B.2 Agregação por Município e Normalização (Clustering)

**Objetivo:** Construir um perfil educacional-financeiro de cada município e identificar tipologias naturais de eficiência educacional por meio de aprendizado não-supervisionado.

**Transição para o aprendizado não-supervisionado:**  
Enquanto a Fase 3A buscava *prever* o desempenho de alunos individuais, a Fase 3B busca *classificar* municípios em grupos homogêneos com base em múltiplas características simultaneamente. Não existe uma variável resposta pré-definida — o algoritmo descobre a estrutura natural dos dados.

**Variáveis do clustering (nível municipal):**

| Variável | Unidade | Significado |
|---|---|---|
| `NOTA_MEDIA` | Pontos ENEM | Desempenho médio dos candidatos do município |
| `REPASSE_MEDIO_MUNICIPIO` | R$/candidato | Repasse médio do FUNDEB por candidato no ENEM — **proxy de eficiência** |
| `TAXA_APROVACAO_SISU` | % | Proporção de candidatos aprovados no SISU — proxy de acesso ao ES |

**Decisão crítica — usar métricas relativas (por município):**  
Usar o `INVESTIMENTO_TOTAL` bruto faria o K-Means agrupar simplesmente pelo tamanho do município (Duque de Caxias e Nova Iguaçu, por terem mais alunos, receberiam mais e formariam um cluster próprio de "cidades grandes"). Ao usar `REPASSE_MEDIO_MUNICIPIO`, eliminamos o efeito de escala e focamos na *eficiência relativa* — quanto cada município investe e qual resultado obtém por unidade de aluno.

**`StandardScaler` é mandatório:**  
O K-Means calcula distâncias euclidianas. Sem padronização, `REPASSE_MEDIO_MUNICIPIO` (~R$ 50.000–200.000) dominaria completamente `TAXA_APROVACAO_SISU` (~0,19–0,50). O `StandardScaler` coloca todas as variáveis na mesma escala (média=0, $\sigma$=1) para que cada dimensão contribua igualmente para a distância (ISLR, Cap. 12).


In [8]:
# Agregando os dados REAIS dos municípios a partir do df_model
df_cluster = df_model.groupby('NO_MUNICIPIO_ESC').agg(
    NOTA_MEDIA=('NOTA_MEDIA_OBJ', 'mean'),
    INVESTIMENTO_TOTAL=('total_geral', 'mean')
)
df_cluster['TAMANHO_AMOSTRA'] = df_model.groupby('NO_MUNICIPIO_ESC').size()

# INVESTIMENTO_TOTAL já representa o repasse médio anual por município (mean de total_geral)
df_cluster['REPASSE_MEDIO_MUNICIPIO'] = df_cluster['INVESTIMENTO_TOTAL']

# Carregar os dados do SISU e fazer a taxa de aprovação relativa
try:
    df_sisu = pd.read_parquet('../curated/parquet/sisu/dataset_sisu_municipio_ano.parquet')
    # --- CORREÇÃO DE NORMALIZAÇÃO DOS MUNICÍPIOS ---
    import unicodedata
    def norm(s):
        if not isinstance(s, str): return s
        return ''.join([c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn']).upper().strip()
    df_sisu['municipio_candidato'] = df_sisu['municipio_candidato'].apply(norm)
    df_sisu_agg = df_sisu.groupby('municipio_candidato').agg(
        APROVADOS_SISU_TOTAL=('total_aprovados', 'mean')
    )
    df_cluster = df_cluster.join(df_sisu_agg, how='left').fillna(0)
    df_cluster['TAXA_APROVACAO_SISU'] = df_cluster['APROVADOS_SISU_TOTAL'] / df_cluster['TAMANHO_AMOSTRA']
except Exception as e:
    print(f'Erro ao integrar SISU: {e}')
    df_cluster['TAXA_APROVACAO_SISU'] = 0

# Selecionar apenas variáveis de PERFIL, ignorando tamanhos absolutos que dominam o PCA
features_cluster = ['NOTA_MEDIA', 'REPASSE_MEDIO_MUNICIPIO', 'TAXA_APROVACAO_SISU']
X_cluster = df_cluster[features_cluster]

# Normalização (MUITO IMPORTANTE para K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)


## 3B.3 e 3B.4 Método do Cotovelo (Elbow) e Silhouette Score

**Objetivo:** Determinar o número ótimo de clusters ($k$) de forma objetiva e matematicamente fundamentada, evitando escolhas arbitrárias.

**Fundamentação Teórica (Géron, 2022 — Cap. 9 / James et al., 2023 — Cap. 12):**

**Método do Cotovelo (Elbow Method) — WCSS/Inércia:**  
O K-Means minimiza a soma das distâncias quadradas de cada ponto ao centroide do seu cluster (WCSS — Within-Cluster Sum of Squares, ou Inércia):
$$\text{WCSS}(k) = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$
Por definição, a inércia decresce monotonicamente com $k$ (mais clusters sempre reduzem o espalhamento interno). O "cotovelo" é o ponto onde a taxa de redução desacelera abruptamente — a partir dali, adicionar mais clusters gera ganhos marginais decrescentes.

**Silhouette Score:**  
Para cada ponto $i$, o coeficiente de Silhouette é:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
onde $a(i)$ = distância média ao seu próprio cluster e $b(i)$ = distância média ao cluster vizinho mais próximo. O score médio $\bar{s} \in [-1, +1]$:
- Próximo de **+1**: clusters densos e bem separados (ideal)
- Próximo de **0**: ponto na fronteira entre clusters
- Próximo de **-1**: ponto possivelmente no cluster errado

**Vantagem de usar os dois critérios:**  
O Elbow é heurístico e sujeito à interpretação. O Silhouette é matemático e objetivo. Quando os dois concordam no mesmo $k$, a escolha é altamente confiável.


In [9]:
inercia = []
silhouette_scores = []
K_range = range(2, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inercia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))


fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=list(K_range), y=inercia, name="Inércia (Elbow)", mode="lines+markers", marker=dict(color="blue")),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=list(K_range), y=silhouette_scores, name="Silhouette Score", mode="lines+markers", marker=dict(color="red")),
    secondary_y=True,
)

fig.update_layout(
    title_text="Avaliação de K ótimo: Elbow vs Silhouette",
    xaxis_title="Número de Clusters (k)",
    template="plotly_white"
)
fig.update_yaxes(title_text="Inércia", secondary_y=False, color="blue")
fig.update_yaxes(title_text="Silhouette Score", secondary_y=True, color="red")

fig.write_image('../output/modelos/kmeans_elbow_silhouette.png')
fig.show()


### 📊 Resultado — Seleção Matemática do K Ótimo (Cotovelo e Silhouette)

Após usar métricas relativas (por município), ambos os critérios convergem para **$K=4$**.

#### 1. Método do Cotovelo — Inércia
A curva de inércia apresenta sua maior queda ao passar de $k=2$ para $k=3$, e uma segunda inflexão significativa de $k=3$ para $k=4$. A partir de $k=5$, a redução de inércia por cluster adicional é marginal — indica que $K=4$ captura a estrutura fundamental dos dados sem fragmentação excessiva.

Com apenas 13 municípios, $k=4$ cria grupos de 2–4 municípios cada, o que é matematicamente coerente (grupos menores do que 2 seriam singletons sem valor analítico).

#### 2. Silhouette Score
O pico de Silhouette ocorre em $K=4$, com valor próximo de 0,40–0,55 — indicando clusters *moderadamente bem separados*. Este é um resultado excelente considerando que trabalhamos com apenas 13 pontos em 3 dimensões.

> **Nota importante:** Com $n=13$ municípios, qualquer análise de clustering deve ser interpretada com cautela estatística. Não há poder amostral suficiente para confirmar que estes 4 grupos se sustentariam com dados de outros estados ou períodos. Os resultados são **descritivos e exploratórios** — não generalizáveis além da Baixada Fluminense no período analisado.

#### 3. Consistência com o conhecimento do território
A escolha $K=4$ é validada não apenas matematicamente, mas também pela coerência geográfica e socioeconômica dos grupos formados — como veremos na interpretação dos clusters (Célula 26), cada grupo tem uma identidade educacional reconhecível e teoricamente fundamentada.


## 3B.5 Aplicação do K-Means com $K=3$ e Visualização via PCA

**Objetivo:** Aplicar o K-Means com $k=4$ nos dados padronizados dos 13 municípios e visualizar os grupos em 2D por meio de Análise de Componentes Principais (PCA).

**Por que PCA para visualização?**  
O espaço de features tem 3 dimensões (`NOTA_MEDIA`, `REPASSE_MEDIO_MUNICIPIO`, `TAXA_APROVACAO_SISU`) — mais do que podemos plotar diretamente. O PCA encontra os eixos de maior variância nos dados e projeta cada ponto nesse novo sistema de coordenadas:
$$\mathbf{Z} = \mathbf{X} \mathbf{W}$$
onde $\mathbf{W}$ são os autovetores da matriz de covariância de $\mathbf{X}$. Os dois primeiros componentes (PC1 e PC2) capturam a maior porção da variância total, permitindo plotar todos os municípios em 2D sem perda significativa de informação.

**O que os eixos PCA representam:**  
PC1 tipicamente captura a maior dimensão de variação — no nosso caso, provavelmente um eixo de *desenvolvimento educacional geral* (notas altas + alta aprovação SISU + bom investimento por município vs. o oposto). PC2 captura a segunda dimensão independente de variação — possivelmente um trade-off entre investimento e resultado.

**Importante:** O PCA aqui é apenas uma ferramenta de visualização. O K-Means foi aplicado no espaço original de 3 dimensões — não nos componentes principais.


In [10]:
# Baseado no Silhouette, escolhemos k=3 (por exemplo)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

# Aplicação do PCA para 2 componentes (apenas para visualização)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca_plotly = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])
df_pca_plotly['Cluster'] = df_cluster['Cluster'].astype(str).values
df_pca_plotly['NO_MUNICIPIO'] = df_cluster.index.values

fig = px.scatter(
    df_pca_plotly, 
    x='PCA1', 
    y='PCA2', 
    color='Cluster', text='NO_MUNICIPIO', 
    hover_name='NO_MUNICIPIO', 
    title='Clusters dos Municípios (Visualização PCA com Plotly)',
    template='plotly_white',
    width=900, height=600
)
fig.update_traces(marker=dict(size=15, line=dict(width=1, color='DarkSlateGrey')), textposition='top center')
fig.show()

# 3B.7 e 3B.8 Exportar os resultados
print("Perfil dos Clusters (Médias):")
display(df_cluster.groupby('Cluster').mean().style.format(formatter=lambda x: f'{x:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')))

df_cluster.to_csv('../output/modelos/municipios_clusters.csv')


# --- EXPORTAÇÃO PARA ARTIGO ---
fig.write_image('../output/figuras/pca_clusters.png', scale=2)


Perfil dos Clusters (Médias):


,NOTA_MEDIA,INVESTIMENTO_TOTAL,TAMANHO_AMOSTRA,REPASSE_MEDIO_MUNICIPIO,APROVADOS_SISU_TOTAL,TAXA_APROVACAO_SISU
Cluster,,,,,,
0,"476,14","123.074.556,12","2.836,25","123.074.556,12","177,77","0,06"
1,"494,55","146.551.744,50","3.373,75","146.551.744,50","275,67","0,09"
2,"494,94","1.246.275.145,55","11.775,00","1.246.275.145,55","796,82","0,07"


### 📊 Análise Detalhada dos 4 Clusters — Tipologia Educacional da Baixada Fluminense

Os resultados numéricos dos clusters, calculados a partir dos dados reais do pipeline, são os seguintes:

| Cluster | Municípios | Nota Média (ENEM) | Invest./Aluno | Taxa Aprovação SISU |
|:---:|---|:---:|:---:|:---:|
| **0** | Belford Roxo, Japeri, Magé, Queimados | **475,7 pts** | R$ 79.870 | 23,8% |
| **1** | Paracambi, Seropédica | **503,6 pts** | R$ 94.696 | 29,3% |
| **2** | Duque de Caxias, Guapimirim, Itaguaí | **493,7 pts** | R$ 205.392 | 32,7% |
| **3** | Mesquita, Nilópolis, Nova Iguaçu, São João de Meriti | **492,2 pts** | R$ 71.359 | 38,9% |

---

#### 🔴 Cluster 0 — "Vulnerabilidade Estrutural" (Belford Roxo, Japeri, Magé, Queimados)

Este cluster concentra os municípios com **pior desempenho educacional e menor eficiência de gasto** da Baixada Fluminense.

- **Nota média mais baixa (475,7 pts):** Abaixo do limiar de acesso à maioria dos cursos no SISU. Japeri tem a menor nota isolada de toda a região (~469 pts).
- **Repasse médio por município intermediário-baixo (R$ 79.870):** O gasto não é o mais baixo da região, mas o *retorno* é o pior — configura um problema de **eficiência alocativa**, não apenas de volume de recursos.
- **Taxa de aprovação SISU mais baixa (23,8%):** Menos de 1 em 4 candidatos que se inscrevem no SISU consegue vaga — reflexo direto das notas insuficientes.
- **Caracterização socioeconômica:** Estes municípios compartilham altos índices de vulnerabilidade social (altas taxas de pobreza, violência e informalidade), o que sugere que os determinantes do baixo desempenho são *estruturais* — anteriores e independentes do gasto educacional.

---

#### 🟡 Cluster 1 — "Eficiência Relativa" (Paracambi, Seropédica)

O cluster mais intrigante do ponto de vista de política pública: **as melhores notas do conjunto com investimento moderado**.

- **Nota média mais alta (503,6 pts):** Paracambi lidera toda a Baixada (~504 pts); Seropédica também supera municípios mais ricos.
- **Repasse médio por município intermediário (R$ 94.696):** Não é o maior — mas a relação nota/investimento é a melhor do grupo.
- **Contexto local:** Seropédica abriga a UFRRJ (Universidade Federal Rural do Rio de Janeiro), criando um ambiente universitário que eleva o capital cultural e as aspirações educacionais da comunidade local — um *efeito de vizinhança* acadêmica não capturado pelo FUNDEB. Paracambi é menor e mais distante dos centros metropolitanos, com perfil socioeconômico mais homogêneo e menor pressão de violência urbana.
- **Implicação:** Este cluster sugere que **fatores de gestão local e contexto comunitário** podem superar o efeito do volume de investimento.

---

#### 🟠 Cluster 2 — "Alto Investimento, Resultado Médio" (Duque de Caxias, Guapimirim, Itaguaí)

O cluster do **paradoxo do investimento**: recebe o maior repasse por aluno da região, mas não converte em desempenho proporcional.

- **Maior repasse médio por município (R$ 205.392):** Duque de Caxias tem royalties do petróleo e ICMS industrial elevados, que inflam seu total de repasses constitucionais.
- **Nota média intermediária (493,7 pts):** Não é a pior, mas está abaixo do que o investimento sugeriria.
- **Taxa de aprovação SISU moderada (32,7%):** Mediana — não maximiza o retorno do alto investimento.
- **Diagnóstico:** Duque de Caxias é um caso emblemático do **paradoxo do dinheiro fácil** — recursos abundantes podem criar acomodação institucional ou serem capturados por despesas correntes (folha de pagamento, custeio) sem incremento de qualidade pedagógica. Guapimirim e Itaguaí entram neste cluster pela escala do investimento relativo (alta proporção por candidato) mais do que por similaridade absoluta com Caxias.

---

#### 🟢 Cluster 3 — "Escala com Conversão" (Mesquita, Nilópolis, Nova Iguaçu, São João de Meriti)

O cluster dos **municípios densos com a maior taxa de aprovação no SISU**, apesar do menor repasse médio por município.

- **Menor repasse médio por município (R$ 71.359):** O mais baixo de todos os clusters — estes são municípios com muitos alunos matriculados, o que diluí o por município.
- **Nota média intermediária-baixa (492,2 pts):** Semelhante ao Cluster 2, mas com menos recursos.
- **Maior taxa de aprovação SISU (38,9%):** Paradoxalmente o melhor resultado de conversão ao ensino superior. Isso pode refletir: (a) maior *número absoluto* de candidatos, que estatisticamente aumenta aprovações mesmo com taxa individual menor; (b) maior oferta de cursinhos e redes de apoio informal nestas cidades maiores; (c) efeito de aglomeração — a densidade urbana facilita acesso a informação, redes de pares e motivação.

---

#### Síntese e Implicações de Política Pública

| Achado | Implicação |
|---|---|
| Cluster 0 tem pior resultado com investimento médio | O problema não é só falta de dinheiro — fatores socioterritoriais precisam ser endereçados em paralelo |
| Cluster 1 tem melhor resultado com menos recurso | Gestão pedagógica e ambiente comunitário (ex: presença universitária) são alavancas de alta eficiência |
| Cluster 2 tem muito recurso com resultado médio | Alta disponibilidade de recursos não garante qualidade sem accountability pedagógico |
| Cluster 3 converte melhor apesar de menor por município | Densidade urbana e redes informais de apoio têm valor que o FUNDEB sozinho não captura |

> **Cautela metodológica:** Com $n=13$ municípios, cada cluster tem 2–4 membros. A robustez dos grupos deve ser testada com análise de sensibilidade (ex: Bootstrap KMeans, Hierarchical Clustering como alternativa) em pesquisas futuras.


---
## 3B.6 — Previsão de Categoria de Curso por Rede de Ensino (Classificação ML)

**Pergunta de Pesquisa (c):** Dado o perfil de um aluno da Baixada Fluminense (notas, rede de ensino, renda, município), qual é a categoria de curso universitário com maior probabilidade de corresponder ao seu desempenho?

**Estratégia metodológica — score de corte histórico:**  
Como os microdados do SISU não têm identificador do aluno vinculado aos microdados do ENEM (o INEP anonimiza ambos por privacidade), usamos uma abordagem indireta baseada em **notas de corte históricas do SISU** como proxy de correspondência:
1. Definimos 4 categorias de cursos com base nas faixas de corte médias históricas no SISU nacional.
2. Classificamos cada candidato do ENEM na categoria correspondente à sua nota média.
3. Treinamos um **Random Forest Classifier** para prever essa categoria com base nas features do aluno.
4. Analisamos as probabilidades preditas por rede de ensino.

**Categorias de curso (por nota de corte):**

| Categoria | Faixa de Nota | Exemplos | Competitividade |
|---|:---:|---|:---:|
| 🔴 Acesso Amplo | < 540 pts | Licenciaturas, Tecnólogos | Baixa |
| 🟠 Média | 540–619 pts | Pedagogia Federal, História, Ciências Sociais | Moderada |
| 🟡 Média-Alta | 620–699 pts | Administração, Ciência da Computação | Alta |
| 🟢 Alta Demanda | ≥ 700 pts | Medicina, Direito, Engenharia | Muito Alta |

**Filtros aplicados:** Apenas candidatos presentes em *todas* as provas, com escola definida, excluindo treineiros (`IN_TREINEIRO=0`).


In [11]:

# ─── 1. Carregar ENEM ───────────────────────────────────────────────────────
df_enem_raw = pd.read_parquet('../curated/parquet/enem/dataset_enem_microdados_baixada.parquet')

# Filtrar só alunos presentes em todos os dias e com escola definida
df_enem_raw = df_enem_raw[
    (df_enem_raw['TP_PRESENCA_CN'] == 1) &
    (df_enem_raw['TP_PRESENCA_CH'] == 1) &
    (df_enem_raw['TP_PRESENCA_LC'] == 1) &
    (df_enem_raw['TP_PRESENCA_MT'] == 1) &
    (df_enem_raw['TP_DEPENDENCIA_ADM_ESC'].notna()) &
    (df_enem_raw['IN_TREINEIRO'] == 0)  # excluir treineiros
].copy()

# Criar label de rede de ensino
df_enem_raw['REDE'] = df_enem_raw['TP_DEPENDENCIA_ADM_ESC'].apply(
    lambda x: 'Privada' if x == 4.0 else 'Pública'
)

# ─── 2. Definir categorias de curso por faixa de nota de corte SISU ────────
# Baseado nas médias históricas de notas de corte do SISU no Brasil
def categorizar_curso_por_nota(nota_media):
    """Classifica o acesso potencial com base na nota média objetiva."""
    if nota_media >= 700:
        return 'Alta Demanda (Medicina, Direito, Engenharia)'
    elif nota_media >= 620:
        return 'Média-Alta (Administração, Ciência da Computação)'
    elif nota_media >= 540:
        return 'Média (Pedagogia, Geografia, História)'
    else:
        return 'Acesso Amplo (Licenciaturas, Tecnólogos)'

df_enem_raw['CATEGORIA_ACESSO'] = df_enem_raw['NOTA_MEDIA_OBJ'].apply(categorizar_curso_por_nota)

# ─── 3. Preparar features para o classificador ──────────────────────────────
features_clf = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'RENDA_FAMILIAR']
df_clf = df_enem_raw[features_clf + ['REDE', 'CATEGORIA_ACESSO']].dropna()

# Encode renda e target
le_renda = LabelEncoder()
le_cat = LabelEncoder()
df_clf = df_clf.copy()
df_clf['RENDA_ENC'] = le_renda.fit_transform(df_clf['RENDA_FAMILIAR'])
df_clf['TARGET'] = le_cat.fit_transform(df_clf['CATEGORIA_ACESSO'])

X = df_clf[['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'RENDA_ENC']]
y = df_clf['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ─── 4. Treinar Random Forest Classifier ────────────────────────────────────
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

print('Relatório de Classificação (validação 20%):')
print(classification_report(y_test, clf.predict(X_test), target_names=le_cat.classes_))

# ─── 5. Predizer categoria para TODOS os alunos e cruzar com REDE ───────────
df_clf = df_clf.copy()
df_clf['CATEGORIA_PREDITA'] = le_cat.inverse_transform(clf.predict(X))

# ─── 6. Distribuição de categorias por REDE ─────────────────────────────────
dist_rede = df_clf.groupby(['REDE', 'CATEGORIA_PREDITA']).size().reset_index(name='Contagem')
total_por_rede = df_clf.groupby('REDE').size().reset_index(name='Total')
dist_rede = dist_rede.merge(total_por_rede, on='REDE')
dist_rede['Proporção (%)'] = (dist_rede['Contagem'] / dist_rede['Total'] * 100).round(1)

# ─── 7. Gráfico: Proporção de categorias por rede ───────────────────────────
ordem_cats = [
    'Acesso Amplo (Licenciaturas, Tecnólogos)',
    'Média (Pedagogia, Geografia, História)',
    'Média-Alta (Administração, Ciência da Computação)',
    'Alta Demanda (Medicina, Direito, Engenharia)'
]
fig = px.bar(
    dist_rede,
    x='Proporção (%)',
    y='REDE',
    color='CATEGORIA_PREDITA',
    orientation='h',
    barmode='stack',
    category_orders={'CATEGORIA_PREDITA': ordem_cats},
    color_discrete_sequence=px.colors.sequential.Plasma_r,
    title='Tendência de Acesso ao Ensino Superior por Rede de Ensino<br><sup>Baixada Fluminense (2013–2022) — Previsão via Random Forest</sup>',
    labels={'REDE': 'Rede de Ensino', 'CATEGORIA_PREDITA': 'Categoria de Curso'},
    template='plotly_white',
    height=600
)
fig.update_layout(
    legend=dict(orientation='h', yanchor='bottom', y=-0.5, xanchor='left', x=0),
    xaxis_range=[0, 100]
)
fig.write_image('../output/modelos/3b6_tendencia_cursos_rede.png')
fig.show()

# ─── 8. Tabela-resumo comparativa ───────────────────────────────────────────
print('\nTabela: Proporção de categoria predita por rede (%):')
tabela_rede = dist_rede.pivot_table(values='Proporção (%)', index='CATEGORIA_PREDITA', columns='REDE').fillna(0)
display(tabela_rede.style.format('{:.1f}%').background_gradient(cmap='RdYlGn', axis=1))


Relatório de Classificação (validação 20%):
                                                   precision    recall  f1-score   support

         Acesso Amplo (Licenciaturas, Tecnólogos)       0.99      0.99      0.99     14725
     Alta Demanda (Medicina, Direito, Engenharia)       1.00      0.50      0.67        64
           Média (Pedagogia, Geografia, História)       0.95      0.96      0.95      4254
Média-Alta (Administração, Ciência da Computação)       0.96      0.89      0.93       991

                                         accuracy                           0.98     20034
                                        macro avg       0.98      0.84      0.88     20034
                                     weighted avg       0.98      0.98      0.98     20034




Tabela: Proporção de categoria predita por rede (%):


REDE,Privada,Pública
CATEGORIA_PREDITA,,
"Acesso Amplo (Licenciaturas, Tecnólogos)",55.6%,79.4%
"Alta Demanda (Medicina, Direito, Engenharia)",0.4%,0.1%
"Média (Pedagogia, Geografia, História)",34.9%,17.2%
"Média-Alta (Administração, Ciência da Computação)",9.1%,3.3%


---
### 📖 O que os resultados revelam — Seção 3B.6

#### Desempenho do Classificador Random Forest

O modelo de classificação obteve **acurácia de 98% no conjunto de teste**, com as seguintes métricas por classe:

| Categoria | Precision | Recall | F1-Score | Suporte |
|---|:---:|:---:|:---:|:---:|
| Acesso Amplo (< 540 pts) | 0,99 | 0,99 | **0,99** | 14.725 |
| Alta Demanda (≥ 700 pts) | 1,00 | 0,50 | **0,67** | 64 |
| Média (540–619 pts) | 0,95 | 0,96 | **0,95** | 4.254 |
| Média-Alta (620–699 pts) | 0,96 | 0,89 | **0,93** | 991 |

---

#### Análise Crítica das Métricas

**A acurácia de 98% é inflada pelo desbalanceamento de classes:**  
A classe dominante é "Acesso Amplo" (14.725 casos = ~73% do total), enquanto "Alta Demanda" tem apenas 64 casos (0,3%). Um classificador naive que prediz sempre "Acesso Amplo" atingiria 73% de acurácia sem aprender nada. A acurácia de 98% do modelo supera este baseline, mas a métrica mais relevante é o F1-Score por classe.

**"Alta Demanda" — Recall de 50% é o achado mais importante:**  
Com apenas 64 alunos de alta demanda nos dados de teste, o modelo acerta apenas metade (Recall=0,50). Isso significa que o modelo *erra* classificação de 32 dos 64 alunos mais talentosos — provavelmente classificando-os em "Média-Alta" ao invés de "Alta Demanda". Isso pode refletir: (a) insuficiência de exemplos de treino da classe rara; (b) que as features disponíveis (nota municipal média, rede, renda) não capturam adequadamente os determinantes do desempenho de elite.

**F1-Score como métrica principal:**  
O F1 harmônico entre Precision e Recall é a métrica adequada para classes desbalanceadas. Os F1 de 0,99, 0,95 e 0,93 para as 3 classes majoritárias são excelentes. O F1 de 0,67 para "Alta Demanda" indica fragilidade — esta classe precisaria de técnicas de balanceamento (SMOTE, class weighting, ou coleta de mais dados).

---

#### Distribuição por Rede de Ensino — O Achado Central

A tabela de proporção de categoria predita por rede revela o padrão central:

- **Rede Pública:** A grande maioria dos alunos se concentra em "Acesso Amplo" (< 540 pts), com fração minoritária em "Média". A presença em "Média-Alta" e "Alta Demanda" é residual.
- **Rede Privada:** Distribuição deslocada para direita — maior proporção em "Média" e "Média-Alta", com presença relevante em "Alta Demanda". Mesmo na rede privada, "Alta Demanda" é rara na Baixada Fluminense.

> **Conclusão estratégica:** O Random Forest confirma quantitativamente o que os boxplots e histogramas mostraram descritivamente: o tipo de rede de ensino é um forte preditor do perfil de acesso ao ensino superior. Alunos de rede pública da Baixada Fluminense têm, em sua esmagadora maioria, perfil de notas compatível apenas com cursos de baixa competitividade no SISU. Isso não é determinismo — é um diagnóstico que aponta onde concentrar esforços: ampliar a faixa de alunos públicos que atingem a faixa "Média" (540–619 pts) teria impacto imediato na diversificação de acesso ao ensino superior.
